In [ ]:
import gc
import pickle
import random
from pathlib import Path

import astropy.coordinates as ac
import healpy as hp
import numpy as np
import pysm3
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.wcs.utils import skycoord_to_pixel
from matplotlib import pyplot as plt
from matplotlib import rcParams
from scipy.stats import spearmanr

import museek.util.tools as tl
from museek.enums.result_enum import ResultEnum

plt.style.use("classic")
plt.subplots_adjust(bottom=0.15, top=0.95, left=0.15, right=0.95)


params = {
    "font.family": "DejaVu Serif",
    "font.serif": "Times New Roman",
    "font.style": "normal",
    "font.weight": "normal",
}
rcParams.update(params)

plt.rcParams["xtick.major.pad"] = "3"
plt.rcParams["ytick.major.pad"] = "2"
plt.rcParams["xtick.labelsize"] = 15
plt.rcParams["ytick.labelsize"] = 15
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"

In [ ]:
# DEFAULT PARAMETERS -- DO NOT REMOVE --
# This cell has "parameters" tag for papermill to recognize the parameters that
# will be overwritten at runtime. Parameters are defined here with their default values.
# When executing the notebook with papermill, a new cell will be injected below this
# cell with the desired parameters being passed to the papermill command, overwritting
# parameter values in this cell. These default parameters can also be inspected by
# calling `papermill --help-notebook <input-notebook>`

block_name: str = "1778715926"  # Block name (CBID)
patch: str = "box14"  # Patch name, e.g. "box14"
base_context_folder: str = "/idia/projects/meerklass/MEERKLASS-1/museek/latest_runs"  # Base context folder. Notebook will look for the data in base_context_folder/patch/block_name/context

In [ ]:
#########  read the calibrated data and raw visibility ##########
# The pickle file is assumed to be in base_context_folder/block_name/block_name/context
context_dir = Path(base_context_folder) / patch / block_name / "context"

with open(context_dir / "aoflagger_plugin_postcalibration.pickle", "rb") as file:
    data_read = pickle.load(file)

# data_read.keys()
calibrated_vis = (
    data_read.get(ResultEnum.CALIBRATED_VIS).result / 10**6.0
)  #### convert from uK to K
freq = (
    data_read.get(ResultEnum.FREQ_SELECT).result / 10**6.0
)  #### convert from Hz to MHz
scan_data = data_read.get(ResultEnum.SCAN_DATA).result
r_vis_synch_ant = data_read.get(ResultEnum.CORRELATION_COEFFICIENT_VIS_SYNCH_ANT).result

del data_read
gc.collect()

initial_flags_before_aoflagger = np.sum(scan_data.flags.array[0:4], axis=0) > 0
initial_flags = scan_data.flags.combine(threshold=1.0)
del scan_data.flags
gc.collect()

freq_all = scan_data.frequencies.squeeze / 10**6.0
timestamps = scan_data.timestamps.array.squeeze()
ra = scan_data.right_ascension.array.squeeze()
dec = scan_data.declination.array.squeeze()

receiver_list = [
    str(receiver) for i_receiver, receiver in enumerate(scan_data.receivers)
]
antenna_list = scan_data._antenna_name_list

receivers = scan_data.receivers
antennas = scan_data.antennas

freqlow_index = np.argmin(
    np.abs(freq_all - 580.0)
)  ####  The calibrated data is from 580. MHz to 1015. MHz
freqhigh_index = np.argmin(np.abs(freq_all - 1015.0))

visibility = np.ma.masked_array(
    scan_data.visibility.array[:, freqlow_index:freqhigh_index, :],
    mask=initial_flags.array[:, freqlow_index:freqhigh_index, :],
    dtype="float32",
)
initial_flags_before_aoflagger = initial_flags_before_aoflagger[
    :, freqlow_index:freqhigh_index, :
]
initial_flags = initial_flags.array[:, freqlow_index:freqhigh_index, :]
del scan_data
gc.collect()

################   if save figures   ######################
# figure_path = data_path+'/'+block_name+'/figures/'
# if not os.path.exists(figure_path):
#    os.makedirs(figure_path)
#    print(f"Directory '{figure_path}' was created.")
# else:
#    print(f"Directory '{figure_path}' already exists.")

In [ ]:
# check the spectra for raw visibility before aoflagger

visibility = np.ma.masked_array(visibility.data, mask=initial_flags_before_aoflagger)
vis_before_aoflagger_timemedian = np.ma.median(visibility, axis=0)
vis_before_aoflagger_freqmedian = np.ma.median(visibility, axis=1)
visibility = np.ma.masked_array(visibility.data, mask=initial_flags)
del initial_flags_before_aoflagger, initial_flags
gc.collect()

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(12, 8))

for i_receiver, receiver in enumerate(receiver_list):
    if "h" in str(receiver_list[i_receiver]):
        ax1.plot(freq, vis_before_aoflagger_timemedian[:, i_receiver])
    elif "v" in str(receiver_list[i_receiver]):
        ax2.plot(freq, vis_before_aoflagger_timemedian[:, i_receiver])


ax1.set_ylabel("Vis before aoflagger HH", fontsize=18)
ax2.set_ylabel("Vis before aoflagger VV", fontsize=18)
ax2.set_xlabel("Frequency [MHz]", fontsize=18)
ax2.set_xlim(freq.min() - 5, freq.max() + 5)
ax1.set_title(block_name + " timemedian before aoflagger HH")
ax2.set_title(block_name + " timemedian before aoflagger VV")
fig.align_ylabels()
# plt.savefig(figure_path+block_name+'_time_median_before_aoflagger.png',dpi=200)
# plt.clf()
plt.show()
plt.close()

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(12, 8))

for i_receiver, receiver in enumerate(receiver_list):
    if "h" in str(receiver_list[i_receiver]):
        ax1.plot(
            (timestamps - timestamps.min()) / 60,
            vis_before_aoflagger_freqmedian[:, i_receiver],
        )
    elif "v" in str(receiver_list[i_receiver]):
        ax2.plot(
            (timestamps - timestamps.min()) / 60,
            vis_before_aoflagger_freqmedian[:, i_receiver],
        )


ax1.set_ylabel("Vis before aoflagger HH", fontsize=18)
ax2.set_ylabel("Vis before aoflagger VV", fontsize=18)
ax2.set_xlabel("Time [min]", fontsize=18)
ax2.set_xlim(0, (timestamps.max() - timestamps.min()) / 60)
ax1.set_title(block_name + " freqmedian before aoflagger HH")
ax2.set_title(block_name + " freqmedian before aoflagger VV")
fig.align_ylabels()
# plt.savefig(figure_path+block_name+'_freq_median_before_aoflagger.png',dpi=200)
# plt.clf()
plt.show()
plt.close()

In [ ]:
# check the waterfall plot for calibrated data and raw visibility for hh and vv

for i_antenna, ant in enumerate(antennas):
    fig = plt.figure(figsize=(12, 3))
    ax = fig.add_subplot(1, 3, 1)
    ax2 = fig.add_subplot(1, 3, 2)
    ax3 = fig.add_subplot(1, 3, 3)

    map1 = ax.pcolormesh(
        freq, (timestamps - timestamps.min()) / 60, calibrated_vis[:, :, i_antenna]
    )

    i_receiver_list = [
        i for i, receiver in enumerate(receivers) if receiver.antenna_name == ant.name
    ]
    for i_receiver in i_receiver_list:
        if "h" in str(receiver_list[i_receiver]):
            map2 = ax2.pcolormesh(
                freq, (timestamps - timestamps.min()) / 60, visibility[:, :, i_receiver]
            )
            ax2.set_title(block_name + " " + str(receiver_list[i_receiver]))
        elif "v" in str(receiver_list[i_receiver]):
            map3 = ax3.pcolormesh(
                freq, (timestamps - timestamps.min()) / 60, visibility[:, :, i_receiver]
            )
            ax3.set_title(block_name + " " + str(receiver_list[i_receiver]))

    ax.set_xlabel("Freq [MHz]", fontsize=16)
    ax.set_ylabel("Time [min]", fontsize=16)
    ax.set_xlim(freq.min(), freq.max())
    ax.set_ylim(0, (timestamps.max() - timestamps.min()) / 60)
    ax.set_title(str(ant.name) + " R_corr=" + str(round(r_vis_synch_ant[i_antenna], 3)))
    cbar_data = plt.colorbar(map1)
    cbar_data.set_label(r"Calibrated Data [$K_{RJ}$]", fontsize=15)

    ax2.set_xlabel("Freq [MHz]", fontsize=16)
    ax2.set_ylabel("Time [min]", fontsize=16)
    ax2.set_xlim(freq.min(), freq.max())
    ax2.set_ylim(0, (timestamps.max() - timestamps.min()) / 60)
    cbar_data = plt.colorbar(map2)
    cbar_data.set_label("Raw Vis", fontsize=15)

    ax3.set_xlabel("Freq [MHz]", fontsize=16)
    ax3.set_ylabel("Time [min]", fontsize=16)
    ax3.set_xlim(freq.min(), freq.max())
    ax3.set_ylim(0, (timestamps.max() - timestamps.min()) / 60)
    cbar_data = plt.colorbar(map3)
    cbar_data.set_label("Raw Vis", fontsize=15)

    plt.tight_layout()
    # plt.savefig(figure_path+block_name+'_'+str(ant.name)+'_waterfall.png',dpi=200)
    # plt.clf()
    plt.show()
    plt.close()

In [ ]:
# check the frequency median of calibrated data

freq_plot_list = np.arange(600, 1020, 100)

for freq_plot in freq_plot_list:
    index_freq_plot = np.argmin(np.abs(freq - freq_plot))
    fig = plt.figure(figsize=(13, 4))

    ax = fig.add_subplot(1, 2, 1)
    ax2 = fig.add_subplot(1, 2, 2)

    #########  randomly select 6 antennas to show, since too many antennas makes the plot unclear   ###########
    selected_antennas = random.sample(antennas, 6)
    i_receiver_list = []
    for ant in selected_antennas:
        i_receiver_list.extend(
            [
                i
                for i, receiver in enumerate(receivers)
                if receiver.antenna_name == ant.name
            ]
        )
    ###########################################################################################################

    for i_receiver in i_receiver_list:
        visibility_plot = visibility[:, index_freq_plot, i_receiver] - np.ma.mean(
            visibility[:, index_freq_plot, i_receiver]
        )

        if "h" in str(receiver_list[i_receiver]):
            ax.plot(
                (timestamps - timestamps.min()) / 60,
                visibility_plot,
                label=receiver_list[i_receiver],
            )
        elif "v" in str(receiver_list[i_receiver]):
            ax2.plot(
                (timestamps - timestamps.min()) / 60,
                visibility_plot,
                label=receiver_list[i_receiver],
            )

    ax.set_xlabel("Time [min]", fontsize=18)
    ax.set_ylabel("Raw Vis - average", fontsize=18)
    ax.set_xlim(0, (timestamps.max() - timestamps.min()) / 60)
    ax.set_title(block_name + " hh " + str(freq_plot) + " MHz")
    ax.legend(loc="upper left", ncol=3, fontsize=10)

    ax2.set_xlabel("Time [min]", fontsize=18)
    ax2.set_ylabel("Raw Vis - average", fontsize=18)
    ax2.set_xlim(0, (timestamps.max() - timestamps.min()) / 60)
    ax2.set_title(block_name + " vv " + str(freq_plot) + " MHz")
    ax2.legend(loc="upper left", ncol=3, fontsize=10)

    plt.tight_layout()
    # plt.savefig(figure_path+block_name+'_raw_minus_average.png',dpi=200)
    # plt.clf()
    plt.show()
    plt.close()

In [ ]:
# check the time median
calibrated_vis_timemedian = np.ma.median(calibrated_vis, axis=0)
visibility_timemedian = np.ma.median(visibility, axis=0)

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, sharex=True, figsize=(12, 12))

for i_receiver, receiver in enumerate(receiver_list):
    if "h" in str(receiver_list[i_receiver]):
        ax1.plot(freq, visibility_timemedian[:, i_receiver])
    elif "v" in str(receiver_list[i_receiver]):
        ax2.plot(freq, visibility_timemedian[:, i_receiver])

for i_antenna, antenna in enumerate(antenna_list):
    ax3.plot(freq, calibrated_vis_timemedian[:, i_antenna])

ax1.set_ylabel("Raw Vis HH", fontsize=18)
ax2.set_ylabel("Raw Vis VV", fontsize=18)
ax3.set_xlabel("Frequency [MHz]", fontsize=18)
ax3.set_xlim(freq.min() - 5, freq.max() + 5)
ax3.set_ylabel(r"Temperature [$K_{RJ}$]", fontsize=18)

ax1.set_title(block_name + " HH")
ax2.set_title(block_name + " VV")
ax3.set_title(block_name + " Calibrated Vis")

fig.align_ylabels()
# plt.savefig(figure_path+block_name+'_time_median.png',dpi=200)
# plt.clf()
plt.show()
plt.close()

In [ ]:
# check the total temperature at 600, 700, 800, 900, 1000 MHz

for i_antenna, antenna in enumerate(antenna_list):
    fig = plt.figure(figsize=(20, 4))

    ax = fig.add_subplot(1, 5, 1)
    freq_plot = 600  # MHz
    index_freq_plot = np.argmin(np.abs(freq - freq_plot))
    sc_data = ax.scatter(
        ra[:, i_antenna],
        dec[:, i_antenna],
        c=np.ma.median(
            calibrated_vis[:, index_freq_plot - 5 : index_freq_plot + 5, i_antenna],
            axis=1,
        ),
        edgecolor="none",
        cmap="jet",
    )
    ax.set_xlabel("RA [Deg]", fontsize=15)
    ax.set_ylabel("DEC [Deg]", fontsize=15)
    ax.set_title(antenna + " " + str(freq_plot) + "MHz")
    ax.set_xlim(np.median(ra) - 25, np.median(ra) + 25)
    ax.set_ylim(np.median(dec) - 8, np.median(dec) + 8)
    cbar_data = plt.colorbar(sc_data)
    cbar_data.set_label(r"Calibrated Data Scatter [$K_{RJ}$]", fontsize=15)

    ax2 = fig.add_subplot(1, 5, 2)
    freq_plot = 700  # MHz
    index_freq_plot = np.argmin(np.abs(freq - freq_plot))
    sc_data = ax2.scatter(
        ra[:, i_antenna],
        dec[:, i_antenna],
        c=np.ma.median(
            calibrated_vis[:, index_freq_plot - 5 : index_freq_plot + 5, i_antenna],
            axis=1,
        ),
        edgecolor="none",
        cmap="jet",
    )
    ax2.set_xlabel("RA [Deg]", fontsize=15)
    ax2.set_ylabel("DEC [Deg]", fontsize=15)
    ax2.set_title(antenna + " " + str(freq_plot) + "MHz")
    ax2.set_xlim(np.median(ra) - 25, np.median(ra) + 25)
    ax2.set_ylim(np.median(dec) - 8, np.median(dec) + 8)
    cbar_data = plt.colorbar(sc_data)
    cbar_data.set_label(r"Calibrated Data Scatter [$K_{RJ}$]", fontsize=15)

    ax3 = fig.add_subplot(1, 5, 3)
    freq_plot = 800  # MHz
    index_freq_plot = np.argmin(np.abs(freq - freq_plot))
    sc_data = ax3.scatter(
        ra[:, i_antenna],
        dec[:, i_antenna],
        c=np.ma.median(
            calibrated_vis[:, index_freq_plot - 5 : index_freq_plot + 5, i_antenna],
            axis=1,
        ),
        edgecolor="none",
        cmap="jet",
    )
    ax3.set_xlabel("RA [Deg]", fontsize=15)
    ax3.set_ylabel("DEC [Deg]", fontsize=15)
    ax3.set_title(antenna + " " + str(freq_plot) + "MHz")
    ax3.set_xlim(np.median(ra) - 25, np.median(ra) + 25)
    ax3.set_ylim(np.median(dec) - 8, np.median(dec) + 8)
    cbar_data = plt.colorbar(sc_data)
    cbar_data.set_label(r"Calibrated Data Scatter [$K_{RJ}$]", fontsize=15)

    ax4 = fig.add_subplot(1, 5, 4)
    freq_plot = 900  # MHz
    index_freq_plot = np.argmin(np.abs(freq - freq_plot))
    sc_data = ax4.scatter(
        ra[:, i_antenna],
        dec[:, i_antenna],
        c=np.ma.median(
            calibrated_vis[:, index_freq_plot - 5 : index_freq_plot + 5, i_antenna],
            axis=1,
        ),
        edgecolor="none",
        cmap="jet",
    )
    ax4.set_xlabel("RA [Deg]", fontsize=15)
    ax4.set_ylabel("DEC [Deg]", fontsize=15)
    ax4.set_title(antenna + " " + str(freq_plot) + "MHz")
    ax4.set_xlim(np.median(ra) - 25, np.median(ra) + 25)
    ax4.set_ylim(np.median(dec) - 8, np.median(dec) + 8)
    cbar_data = plt.colorbar(sc_data)
    cbar_data.set_label(r"Calibrated Data Scatter [$K_{RJ}$]", fontsize=15)

    ax5 = fig.add_subplot(1, 5, 5)
    freq_plot = 1000  # MHz
    index_freq_plot = np.argmin(np.abs(freq - freq_plot))
    sc_data = ax5.scatter(
        ra[:, i_antenna],
        dec[:, i_antenna],
        c=np.ma.median(
            calibrated_vis[:, index_freq_plot - 5 : index_freq_plot + 5, i_antenna],
            axis=1,
        ),
        edgecolor="none",
        cmap="jet",
    )
    ax5.set_xlabel("RA [Deg]", fontsize=15)
    ax5.set_ylabel("DEC [Deg]", fontsize=15)
    ax5.set_title(antenna + " " + str(freq_plot) + "MHz")
    ax5.set_xlim(np.median(ra) - 25, np.median(ra) + 25)
    ax5.set_ylim(np.median(dec) - 8, np.median(dec) + 8)
    cbar_data = plt.colorbar(sc_data)
    cbar_data.set_label(r"Calibrated Data Scatter [$K_{RJ}$]", fontsize=15)

    plt.tight_layout()
    # plt.savefig(figure_path+block_name+'_'+antenna+'_waterfall.png',dpi=200)
    # plt.clf()
    plt.show()
    plt.close()

In [ ]:
##########  compare with synch model (time median removed for both calibrated data and synch model ) ###########

nside = 128  # resolution parameter at which the synchrotron model is to be calculated
beamsize = 57.5  # the beam fwhm used to smooth the Synch model [arcmin]
beam_frequency = 1500.0  # reference frequency at which the beam fwhm are defined [MHz]
sky = pysm3.Sky(
    nside=nside, preset_strings=["s1"]
)  # Here we use "s1" vresion of synch model, please see https://pysm3.readthedocs.io/en/latest/models.html#synchrotron

calibrated_vis_timemedian = np.ma.median(calibrated_vis, axis=0)

freq_plot_list = np.arange(600, 1020, 100)
for i_antenna, antenna in enumerate(antenna_list):
    for freq_plot in freq_plot_list:
        index_freq_plot = np.argmin(np.abs(freq - freq_plot))

        if calibrated_vis[:, index_freq_plot, i_antenna].mask.all():
            print(antenna + " " + str(freq_plot) + "MHz is masked")
        else:
            #########   produce synch model at a certain frequency and smooth   #########
            map_reference = sky.get_emission(freq_plot * u.MHz).value
            map_reference_smoothed = pysm3.apply_smoothing_and_coord_transform(
                map_reference,
                fwhm=beamsize
                * u.arcmin
                * ((beam_frequency * u.MHz) / (freq_plot * u.MHz)).decompose().value,
            )

            #########   map the smoothed synch model to the same sky covered by ra,dec of scan_data
            c = SkyCoord(
                ra=ra[:, i_antenna] * u.degree,
                dec=dec[:, i_antenna] * u.degree,
                frame="icrs",
            )
            theta = 90.0 - (c.galactic.b / u.degree).value
            phi = (c.galactic.l / u.degree).value
            synch_I = hp.pixelfunc.get_interp_val(
                map_reference_smoothed[0], theta / 180.0 * np.pi, phi / 180.0 * np.pi
            )
            synch_I = np.ma.masked_array(
                synch_I, mask=calibrated_vis[:, index_freq_plot, i_antenna].mask
            )
            synch_I = synch_I / 10**6.0

            ########  plot  #########
            fig = plt.figure(figsize=(12, 3))
            ax = fig.add_subplot(1, 2, 1)
            ax2 = fig.add_subplot(1, 2, 2)

            calibrated_vis_nomedian = (
                calibrated_vis[:, index_freq_plot, i_antenna]
                - calibrated_vis_timemedian[np.newaxis, index_freq_plot, i_antenna]
            )
            sc_data = ax.scatter(
                ra[:, i_antenna],
                dec[:, i_antenna],
                c=calibrated_vis_nomedian,
                edgecolor="none",
                cmap="jet",
                vmin=-0.5,
                vmax=1.0,
            )
            ax.set_xlabel("RA [Deg]", fontsize=15)
            ax.set_ylabel("DEC [Deg]", fontsize=15)
            ax.set_title(antenna + " " + str(freq_plot) + "MHz")
            ax.set_xlim(np.median(ra) - 25, np.median(ra) + 25)
            ax.set_ylim(np.median(dec) - 8, np.median(dec) + 8)
            cbar_data = plt.colorbar(sc_data)
            cbar_data.set_label(r"Calibrated Data [$K_{RJ}$]", fontsize=15)

            synch_I_nomedian = synch_I - np.ma.median(synch_I)
            spearman_corr, spearman_p = spearmanr(
                calibrated_vis_nomedian.data[~calibrated_vis_nomedian.mask],
                synch_I_nomedian.data[~calibrated_vis_nomedian.mask],
            )
            sc_data = ax2.scatter(
                ra[:, i_antenna],
                dec[:, i_antenna],
                c=synch_I_nomedian,
                edgecolor="none",
                cmap="jet",
                vmin=-0.5,
                vmax=1.0,
            )
            ax2.set_xlabel("RA [Deg]", fontsize=15)
            ax2.set_ylabel("DEC [Deg]", fontsize=15)
            ax2.set_title("correlation coefficient = " + str(round(spearman_corr, 4)))
            ax2.set_xlim(np.median(ra) - 25, np.median(ra) + 25)
            ax2.set_ylim(np.median(dec) - 8, np.median(dec) + 8)
            cbar_data = plt.colorbar(sc_data)
            cbar_data.set_label(r"Synch [$K_{RJ}$]", fontsize=15)

            plt.tight_layout()
            # plt.savefig(figure_path+block_name+'_'+antenna+'_'+str(freq_plot) +'MHz.png',dpi=200)
            # plt.clf()
            plt.show()
            plt.close()

In [ ]:
#############     Below is the module for map-making      ################

In [ ]:
# define the wcs for map. For now, different block have different wcs

import numpy as np
from astropy.wcs import WCS

# Create a new WCS object with two dimensions
wcs = WCS(naxis=2)

pix_reso = 0.5  #  map resolution deg

x_min = np.median(ra) - 25.0  # map region
x_max = np.median(ra) + 25.0  # map region
y_min = np.median(dec) - 10.0  # map region
y_max = np.median(dec) + 10.0  # map region

crpix_x = int((x_max - x_min) / pix_reso / 2.0)
crpix_y = int((y_max - y_min) / pix_reso / 2.0)

# Assuming the RA ad Dec axis are the first and second axis respectively
# Define the reference pixel (center of the array or some other reference)
wcs.wcs.crpix = [crpix_x, crpix_y]  # Middle of the array
wcs.wcs.cdelt = np.array([pix_reso, pix_reso])  # degrees/pixel for RA and Dec
wcs.wcs.crval = [(x_max + x_min) / 2.0, (y_max + y_min) / 2.0]
wcs.wcs.ctype = ["RA---ZEA", "DEC--ZEA"]  # Use ZEA projection for RA and Dec

# Optionally set unit types
wcs.wcs.cunit = ["deg", "deg"]

# Display the WCS header to check
print("wcs", wcs)

# set the shape of map
map_shape = (
    np.round((x_max - x_min) / pix_reso).astype(int) + 1,
    np.round((y_max - y_min) / pix_reso).astype(int) + 1,
)

In [ ]:
# project the calibrated data on the defined 2D grid, combining all antennas

map_making_data = np.ones((map_shape[0], map_shape[1], len(freq)))
hit_data = np.ones((map_shape[0], map_shape[1], len(freq)))

sky_sc = ac.SkyCoord(
    ra=ra.flatten() * u.deg, dec=dec.flatten() * u.deg
)  # pointings in observation
pix_coords = skycoord_to_pixel(sky_sc, wcs)

for i_freq, freq_value in enumerate(freq):
    data = calibrated_vis[:, i_freq, :] - np.ma.median(
        calibrated_vis[:, i_freq, :], axis=0, keepdims=True
    )
    output, weight, hit, xedges, yedges = tl.project_2d(
        pix_coords[1], pix_coords[0], data.flatten(), map_shape, weights=None
    )
    map_making_data[:, :, i_freq] = output
    hit_data[:, :, i_freq] = hit

map_making_data = np.ma.masked_array(map_making_data, mask=np.isnan(map_making_data))
hit_data = np.ma.masked_array(hit_data, mask=np.isnan(map_making_data))

x_pix_coords = xedges[1:] - np.diff(xedges)[0] / 2.0
y_pix_coords = yedges[1:] - np.diff(yedges)[0] / 2.0
x_sky_coords = wcs.wcs.crval[0] + (x_pix_coords - wcs.wcs.crpix[0]) * wcs.wcs.cdelt[0]
y_sky_coords = wcs.wcs.crval[1] + (y_pix_coords - wcs.wcs.crpix[1]) * wcs.wcs.cdelt[1]

In [ ]:
#######  save map-making output   ##########

# save the map-making results
x_sky_coords_mesh, y_sky_coords_mesh = np.meshgrid(x_sky_coords, y_sky_coords)
arrays_dict = {
    "map": map_making_data,
    "hit": hit_data,
    "wcs": wcs,
    "ra": x_sky_coords_mesh,
    "dec": y_sky_coords_mesh,
    "freq": freq,
    "antenna_list": antenna_list,
}

with open(data_dir / "_map_making.pkl", "wb") as f:
    pickle.dump(arrays_dict, f)

In [ ]:
#######   map-making for synch and compared with scan data, all antennas combined  #############

nside = 128  # resolution parameter at which the synchrotron model is to be calculated
beamsize = 57.5  # the beam fwhm used to smooth the Synch model [arcmin]
beam_frequency = 1500.0  # reference frequency at which the beam fwhm are defined [MHz]
sky = pysm3.Sky(
    nside=nside, preset_strings=["s1"]
)  # Here we use "s1" vresion of synch model, please see https://pysm3.readthedocs.io/en/latest/models.html#synchrotron

freq_plot_list = np.arange(600, 1020, 100)

for freq_plot in freq_plot_list:
    index_freq_plot = np.argmin(np.abs(freq - freq_plot))

    if map_making_data[:, :, index_freq_plot].mask.all():
        print(str(freq_plot) + "MHz is masked")
    else:
        #########   produce synch model at a certain frequency and smooth   #########
        map_reference = sky.get_emission(freq_plot * u.MHz).value
        map_reference_smoothed = pysm3.apply_smoothing_and_coord_transform(
            map_reference,
            fwhm=beamsize
            * u.arcmin
            * ((beam_frequency * u.MHz) / (freq_plot * u.MHz)).decompose().value,
        )

        #########   map the smoothed synch model to the same sky covered by ra,dec of scan_data
        c = SkyCoord(
            ra=ra.flatten() * u.degree, dec=dec.flatten() * u.degree, frame="icrs"
        )
        theta = 90.0 - (c.galactic.b / u.degree).value
        phi = (c.galactic.l / u.degree).value
        synch_I = hp.pixelfunc.get_interp_val(
            map_reference_smoothed[0], theta / 180.0 * np.pi, phi / 180.0 * np.pi
        )
        synch_I = np.ma.masked_array(
            synch_I, mask=calibrated_vis[:, index_freq_plot, :].mask.flatten()
        )
        synch_I = synch_I / 10**6.0

        sky_sc = ac.SkyCoord(
            ra=ra.flatten() * u.deg, dec=dec.flatten() * u.deg
        )  # pointings in observation
        pix_coords = skycoord_to_pixel(sky_sc, wcs)

        data = synch_I - np.ma.median(synch_I, axis=0, keepdims=True)
        output, weight, hit, xedges, yedges = tl.project_2d(
            pix_coords[1], pix_coords[0], data.flatten(), map_shape, weights=None
        )
        map_making_synch = np.ma.masked_array(output, mask=np.isnan(output))

        fig = plt.figure(figsize=(15, 3))

        ax = fig.add_subplot(1, 3, 1)
        mk_data = plt.pcolormesh(
            x_sky_coords,
            y_sky_coords,
            map_making_data[:, :, index_freq_plot].T,
            vmax=1.0,
            vmin=-0.5,
        )
        ax.set_xlabel("RA [Deg]", fontsize=15)
        ax.set_ylabel("DEC [Deg]", fontsize=15)
        ax.set_title(block_name + " " + str(freq_plot) + "MHz")
        ax.set_xlim(x_sky_coords.min(), x_sky_coords.max())
        ax.set_ylim(y_sky_coords.min(), y_sky_coords.max())
        cbar_mp = plt.colorbar(mk_data)
        cbar_mp.set_label(r"Map Making [$K_{RJ}$]", fontsize=15)

        ax2 = fig.add_subplot(1, 3, 2)
        mk_data = plt.pcolormesh(
            x_sky_coords, y_sky_coords, map_making_synch.T, vmax=1.0, vmin=-0.5
        )
        ax2.set_xlabel("RA [Deg]", fontsize=15)
        ax2.set_ylabel("DEC [Deg]", fontsize=15)
        ax2.set_xlim(x_sky_coords.min(), x_sky_coords.max())
        ax2.set_ylim(y_sky_coords.min(), y_sky_coords.max())
        cbar_mp = plt.colorbar(mk_data)
        cbar_mp.set_label(r"Synch [$K_{RJ}$]", fontsize=15)

        ax3 = fig.add_subplot(1, 3, 3)
        mk_data = plt.pcolormesh(
            x_sky_coords, y_sky_coords, hit_data[:, :, index_freq_plot].T
        )
        ax3.set_xlabel("RA [Deg]", fontsize=15)
        ax3.set_ylabel("DEC [Deg]", fontsize=15)
        ax3.set_xlim(x_sky_coords.min(), x_sky_coords.max())
        ax3.set_ylim(y_sky_coords.min(), y_sky_coords.max())
        cbar_mp = plt.colorbar(mk_data)
        cbar_mp.set_label("HIT", fontsize=15)

        plt.tight_layout()
        # plt.savefig(figure_path+block_name+'_'+antenna+'_'+str(freq_plot) +'MHz_mapmaking.png',dpi=200)
        # plt.clf()
        plt.show()
        plt.close()